<a href="https://colab.research.google.com/github/milleau98/2026-gig-data-analysis/blob/main/notebooks/dataset_generators/google_trends_pull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pytrends pandas


In [2]:
from pytrends.request import TrendReq
import pandas as pd
import time

# connect to Google trends
pytrends = TrendReq(hl='en-US', tz=360)

anchor = 'Uber'

keyword_groups = [
    [anchor,'Lyft','DoorDash','Instacart','Grubhub'],
    [anchor, 'Fiverr','Upwork','side hustle','gig'],
    [anchor, 'Herbalife','Primerica','Tupperware','Avon']
]

all_data = []

for i, group in enumerate(keyword_groups):
  # timeframe and geo
  pytrends.build_payload(group, timeframe='2010-01-01 2025-12-31', geo='US')

  df=pytrends.interest_over_time()

  df=df.drop(columns=['isPartial'])

  # store anchor values
  if i == 0:
    anchor_series = df[anchor]
  else:
    scale_factor = anchor_series.div(df[anchor]).replace([float("inf"), -float("inf")], 0)
    df = df.multiply(scale_factor, axis=0)


  all_data.append(df.drop(columns=[anchor]))

  time.sleep(15)

# combine groups of data
final_data = pd.concat(all_data, axis=1)

final_data.head()

,Lyft,DoorDash,Instacart,Grubhub,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Tupperware,Avon
date,,,,,,,,,,,,
2010-01-01,0,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,3.0,21.0
2010-02-01,0,0,0,1,0.0,0.0,0.0,7.0,3.0,2.0,3.0,21.0
2010-03-01,0,0,0,0,0.0,0.0,0.0,7.0,3.0,3.0,3.0,20.0
2010-04-01,0,0,0,0,0.0,0.0,0.0,7.0,3.0,3.0,2.0,21.0
2010-05-01,0,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,3.0,23.0


In [3]:
# convert weekly trends to monthly averages
monthly = final_data.resample("M").mean()

monthly['year'] = monthly.index.year
monthly['month'] = monthly.index.month
monthly['date'] = monthly.index

monthly = monthly.reset_index(drop=True)

monthly.head()

/tmp/ipykernel_5022/1836030185.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = final_data.resample("M").mean()


,Lyft,DoorDash,Instacart,Grubhub,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Tupperware,Avon,year,month,date
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,3.0,21.0,2010,1,2010-01-31
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,7.0,3.0,2.0,3.0,21.0,2010,2,2010-02-28
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,3.0,20.0,2010,3,2010-03-31
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,2.0,21.0,2010,4,2010-04-30
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,3.0,23.0,2010,5,2010-05-31


In [4]:
# convert weekly trends into quarterly averages
quarterly = final_data.resample("Q").mean()
quarterly.index = quarterly.index.to_period("Q").to_timestamp()
quarterly['year'] = quarterly.index.year
quarterly['quarter'] = quarterly.index.quarter
quarterly['date'] = quarterly.index
quarterly = quarterly.reset_index(drop=True)
quarterly.head()

/tmp/ipykernel_5022/3960241845.py:2: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  quarterly = final_data.resample("Q").mean()


,Lyft,DoorDash,Instacart,Grubhub,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Tupperware,Avon,year,quarter,date
0,0.0,0.0,0.0,0.333333,0.0,0.0,0.0,7.000000,3.000000,2.333333,3.000000,20.666667,2010,1,2010-01-01
1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,7.333333,3.000000,2.333333,2.666667,22.333333,2010,2,2010-04-01
2,0.0,0.0,0.0,0.333333,0.0,0.0,0.0,7.666667,3.000000,2.000000,3.000000,22.666667,2010,3,2010-07-01
3,0.0,0.0,0.0,1.000000,0.0,0.0,0.0,8.000000,2.666667,2.000000,3.000000,21.000000,2010,4,2010-10-01
4,0.0,0.0,0.0,1.000000,0.0,0.0,0.0,7.666667,3.000000,2.000000,2.333333,19.333333,2011,1,2011-01-01


In [6]:
# convert weekly trends into annual averages
annual = final_data.resample("Y").mean()
annual.index = annual.index.to_period("Y").to_timestamp()
annual["year"] = annual.index.year
annual["date"] = annual.index
annual = annual.reset_index(drop=True)

annual.head()

/tmp/ipykernel_5022/646657277.py:2: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  annual = final_data.resample("Y").mean()


,Lyft,DoorDash,Instacart,Grubhub,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Tupperware,Avon,year,date
0,0.000000,0.0,0.00,0.416667,0.000000,0.0,0.0,7.500000,2.916667,2.166667,2.916667,21.666667,2010,2010-01-01
1,0.000000,0.0,0.00,1.083333,0.166667,0.0,0.0,7.666667,3.333333,2.083333,2.333333,19.750000,2011,2011-01-01
2,0.000000,0.0,0.00,2.333333,1.083333,0.0,0.0,6.750000,4.833333,1.916667,2.666667,19.333333,2012,2012-01-01
3,1.333333,0.0,0.00,3.833333,1.000000,0.0,0.0,6.416667,7.000000,1.916667,3.000000,18.583333,2013,2013-01-01
4,4.583333,0.0,0.75,7.083333,1.083333,0.0,0.0,6.500000,7.333333,1.916667,3.000000,17.416667,2014,2014-01-01


In [7]:
import os
os.makedirs("data/google_trends", exist_ok=True)

# create dataset files
monthly.to_csv('data/google_trends/google_trends_monthly.csv', index=False)
quarterly.to_csv('data/google_trends/google_trends_quarterly.csv', index=False)
annual.to_csv("data/google_trends/google_trends_annual.csv", index=False)

In [8]:
# download as csv
from google.colab import files
files.download('data/google_trends/google_trends_monthly.csv')
files.download('data/google_trends/google_trends_quarterly.csv')
files.download('data/google_trends/google_trends_annual.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>